# Weighted geometric median pixel composites <img align="right" src="../Supplementary_data/dea_logo.jpg">

* **[Sign up to the DEA Sandbox](https://app.sandbox.dea.ga.gov.au/)** to run this notebook interactively from a browser
* **Compatibility:** Notebook currently compatible with both the `NCI` and `DEA Sandbox` environments
* **Products used:** 
[ga_s2am_ard_3](https://explorer.dea.ga.gov.au/products/ga_s2am_ard_3), [ga_s2bm_ard_3](https://explorer.dea.ga.gov.au/products/ga_s2bm_ard_3)

## Background

The `geomedian` is a robust compositing technique that summarises a time series of satellite observations into a single representative image. Unlike a per-band median, the geomedian preserves the spectral relationships between bands, producing physically realistic composites that are less sensitive to clouds, shadows and other outliers. This approach underpins the [annual Landsat GeoMAD](https://knowledge.dea.ga.gov.au/data/product/dea-geometric-median-and-median-absolute-deviation-landsat/?tab=description) products produced by Digital Earth Australia (DEA).

A _weighted_ geomedian extends this concept by assigning greater importance to observations with particular spectral characteristics. In this notebook, weights are derived from a normalised difference index calculated from the NIR and Red bands (i.e. NDVI). This allows the compositing process to favour observations that represent specific landscape conditions.

## Description
In this notebook we will explore how to generate weighted geomedians by creating weighted composites for an example region, and compare this with the standard geomedian approach. This comparison demonstrates how weighting can be used to emphasise different aspects of the landscape while maintaining the benefits of geomedian compositing.

### References
* Roberts, D., Mueller, N., & Mcintyre, A. (2017). High-dimensional pixel composites from earth observation time series. IEEE Transactions on Geoscience and Remote Sensing, 55(11), 6254-6264.

### Load packages

In [ ]:
import datacube
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from odc.algo import geomedian_with_mads
from dea_tools.datahandling import load_ard
from dea_tools.dask import create_local_dask_cluster

### Set up a dask cluster

This will help keep our memory use down and conduct the analysis in parallel. If you'd like to view the `dask dashboard`, click on the hyperlink that prints below the cell. You can use the dashboard to monitor the progress of calculations.


In [ ]:
create_local_dask_cluster(threads_per_worker=6)

## Connect to the datacube

In [ ]:
dc = datacube.Datacube(app='Generating_weighted_geomedian_composites')

## Load Sentinel-2 from the datacube

For the default example, we will load data over a lawn farm in Canberra, ACT.  This is a good example location because the lawn is repeatedly grown and harvested throughout the year so alternates from very green to bare ground.

In [ ]:
# Set up centre of area of interest, and area to buffer coordinates by
lat, lon = -35.31316, 149.17085 # lawn farm in canberra 
buffer = 0.008
time_range = ('2024')
bands = ['nbart_green','nbart_red','nbart_blue', 'nbart_nir_1']

In [ ]:
# Create a reusable query
query = {
    'x': (lon - buffer, lon + buffer),
    'y': (lat + buffer, lat - buffer),
    'time': time_range,
    'measurements': bands,
    'resolution': (-10, 10),
    'group_by': 'solar_day',
    'output_crs': 'EPSG:3577'
}

# Load available data
ds = load_ard(dc=dc, 
              products=['ga_s2am_ard_3', 'ga_s2bm_ard_3'],
              cloud_mask='s2cloudless',
              dask_chunks={},
              **query)

## Function for weighted GM

In [ ]:
import dask
import dask.array as da
import functools
from geomad import nanwgeomedian_pcm
from odc.algo import yxbt_sink, reshape_yxbt
from odc.algo import randomize
from typing import Literal

def xr_weighted_geomedian(
    src: xr.Dataset | xr.DataArray,
    band1: str = "nbart_nir",
    band2: str = "nbart_red",
    rho: float = 6.0,
    delta: float = 1.0,
    xi: float = 0.0,
    alpha: float | None = None,
    gamma: float | None = None,
    beta: float | None = None,
    sigma: float | None = None,
    out_chunks: tuple[int, int, int] | None = None,
    reshape_strategy: Literal["mem", "yxbt"] = "yxbt",
    eps: float = 0.0001,
    maxiters: int = 1000,
    num_threads: int = 1,
    **kw,
) -> xr.Dataset:
    """
    Compute a weighted geomedian composite from a time series of Earth observation data.

    This function applies Geoscience Australia's weighted geomedian algorithm
    (`geomad.nanwgeomedian_pcm`) to a Dask-backed xarray Dataset or DataArray.
    The implementation extends the standard geomedian approach by assigning a
    weight to each observation based on a normalised difference index calculated
    from two nominated bands (typically NIR and Red - NDVI). This allows the composite
    to preferentially select observations with particular spectral
    characteristics.
    
    The underlying geomedian implementation is based on the pixel composite
    method described by Roberts et al. (2017), which uses a geometric median
    in spectral space to generate robust Earth observation composites.

    Parameters
    ----------
    src : xr.Dataset or xr.DataArray
        Input data as a Dask-backed xarray object containing a temporal stack
        of observations. When a Dataset is supplied, bands are reshaped into
        the YXBT format expected by the geomedian implementation
        (Y=rows, X=columns, B=bands, T=time). 
        
    band1 : str, default="nbart_nir"
        Name of the first band used to calculate the weighting index.
        Typically the near-infrared band.

    band2 : str, default="nbart_red"
        Name of the second band used to calculate the weighting index.
        Typically the red band.

    rho : float, default=6.0
        Weighting strength parameter passed to
        ``geomad.nanwgeomedian_pcm``. Larger values increase the influence
        of the weighting index on compositing behaviour, resulting in stronger
        preference for observations with extreme index values.

    delta : float, default=1.0
        Scaling parameter controlling the amplitude of the weighting function.
        Larger values increase the separation between highly weighted and
        weakly weighted observations.

    xi : float, default=0.0
        Centre point of the weighting function. For NDVI-based weighting,
        this determines the index value at which observations transition
        from being down-weighted to up-weighted.

    alpha : float, optional
        Shape parameter controlling the lower portion of the weighting
        curve. 

    gamma : float, optional
        Shape parameter controlling the upper portion of the weighting
        curve.

    beta : float, optional
        Additional weighting function parameter passed directly to the
        underlying PCM implementation. 
        
    sigma : float, optional
        Smoothing or spread parameter for the weighting function.
        Larger values generally produce a more gradual transition between
        low-weight and high-weight observations.

    out_chunks : tuple[int, int, int], optional
        If supplied, rechunk the output array after computation using
        ``(y, x, band)`` chunk sizes.

    reshape_strategy : {"mem", "yxbt"}, default="yxbt"
        Strategy used to reshape an input Dataset into the YXBT structure
        required by the geomedian algorithm.

        - ``"mem"``: Faster when sufficient memory is available on a
          single-worker system.
        - ``"yxbt"``: More scalable and suitable for multi-worker Dask
          deployments.

    eps : float, default=0.0001
        Convergence tolerance for the iterative geomedian solver.
        Iteration stops when successive estimates differ by less than this
        value.

    maxiters : int, default=1000
        Maximum number of iterations permitted for the geomedian solver
        for each output pixel.

    num_threads : int, default=1
        Number of worker threads used internally by the geomedian algorithm.
        When using Dask, a value of 1 is usually appropriate because
        parallelism is typically provided by Dask itself.

    Returns
    -------
    xr.Dataset
        An xarray Dataset containing the weighted geomedian composite for each
        input band.

    References
    ----------
    Roberts, D., Mueller, N., & Mcintyre, A. (2017). High-dimensional pixel
    composites from earth observation time series. IEEE Transactions
    on Geoscience and Remote Sensing, 55(11), 6254-6264.
    
    """

    # Geomedian implementation requires Dask-backed inputs.
    if not dask.is_dask_collection(src):
        raise ValueError("This method only works on Dask inputs")

    # Convert Dataset inputs into the YXBT layout expected by geomad.
    if isinstance(src, xr.DataArray):
        yxbt = src
    else:
        ny, nx = kw.get("work_chunks", (100, 100))

        if reshape_strategy == "mem":
            yxbt = yxbt_sink(src, (ny, nx, -1, -1))
        elif reshape_strategy == "yxbt":
            yxbt = reshape_yxbt(src, yx_chunks=(ny, nx))
        else:
            raise ValueError(
                f"Reshape strategy '{reshape_strategy}' not understood "
                "use one of: mem or yxbt"
            )

    _, _, nb, _ = yxbt.shape

    assert yxbt.chunks is not None

    # Geomedian expects time and band dimensions to be contained
    # within a single Dask block.
    if yxbt.data.numblocks[2:4] != (1, 1):
        raise ValueError(
            "There should be one dask block along time and band dimension"
        )

    chunks = (*yxbt.chunks[:2], (nb,))

    # Find the index values of the bands requested for weighting
    try:
        band_index = yxbt.get_index("band")
        bi = band_index.get_loc(band1)
        bj = band_index.get_loc(band2)
    
    except KeyError as e:
        raise ValueError(
            f"Band '{e.args[0]}' not found. "
            f"Available bands: {list(band_index)}"
        ) from None

    # Configure the weighted geomedian operator. Band indices define
    # the normalised-difference weighting metric (e.g. NDVI).
    op = functools.partial(
        nanwgeomedian_pcm,
        bi=bi,
        bj=bj,
        rho=rho,
        delta=delta,
        xi=xi,
        alpha=alpha,
        gamma=gamma,
        beta=beta,
        sigma=sigma,
        eps=eps,
        maxiters=maxiters,
        num_threads=num_threads,
    )

    # Apply the weighted geomedian independently to each spatial block.
    _wgm = da.map_blocks(
        op,
        yxbt.data,
        dtype="float32",
        drop_axis=3,
        chunks=chunks,
        name=randomize("wgeomedian"),
    )

    if out_chunks is not None:
        _wgm = _wgm.rechunk(out_chunks)

    # Reconstruct xarray coordinates and metadata, dropping the
    # temporal dimension collapsed during compositing.
    drop_dim = yxbt.dims[3]
    dims = yxbt.dims[:3]

    coords = {
        coord_name: coord
        for coord_name, coord in yxbt.coords.items()
        if drop_dim not in coord.dims
    }

    # Construct xarray
    result = xr.DataArray(
        data=_wgm[:, :, :nb].astype(yxbt.dtype),
        dims=dims,
        coords=coords,
        attrs=yxbt.attrs,
    ).to_dataset("band")

    # Propagate source metadata to each output band.
    for dv in result.data_vars.values():
        dv.attrs.update(yxbt.attrs)

    return result

## Compute weighted geomedian composites

* `rho`: This number controls how much the function will weight the band index (e.g. NDVI). In the case of NDVI, a negative `rho` value will result in weighting towards low vegetation cover, and a positive value will weight towards higher vegetation cover.

In [ ]:
rho=6

# so data doesn't need to load twice make it persist on the cluster
ds = ds.persist() 

# compute greener and barer composites
wgm_green = xr_weighted_geomedian(ds, rho=rho, band1='nbart_nir_1', band2='nbart_red').load()
wgm_bare = xr_weighted_geomedian(ds, rho=-rho, band1='nbart_nir_1', band2='nbart_red').load()

## Compute standard geomedian

We'll use this to compare with our weighted geomedians

In [ ]:
gm = geomedian_with_mads(
    ds,
    reshape_strategy='yxbt',
    compute_mads=False,
    compute_count=False
).load()

## Plot our results

Below we will plot the pixel composites as true colour ('RGB') images, along with NDVI. The standard geomedian (left) provides a representative view of the landscape by treating all observations equally. By introducing NDVI-based weighting, the composite can be biased towards either greener or barer surface conditions.

The effect is particularly visible in the large centre-pivot irrigation fields, which in this case are used for turf (lawn) production. The greener composite preferentially selects observations when the turf is actively growing, while the barer composite favours periods when vegetation cover is reduced, exposing more of the underlying soil surface.

### True Colour

In [ ]:
fig,ax=plt.subplots(1,3, figsize=(15,5), sharey=True, layout='constrained')
gm[['nbart_red', 'nbart_green', 'nbart_blue']].to_array().plot.imshow(ax=ax[0],robust=True, add_labels=False);
wgm_green[['nbart_red', 'nbart_green', 'nbart_blue']].to_array().plot.imshow(ax=ax[1], robust=True, add_labels=False);
wgm_bare[['nbart_red', 'nbart_green', 'nbart_blue']].to_array().plot.imshow(ax=ax[2], robust=True, add_labels=False);
ax[0].set_title(f'Standard geomedian')
ax[1].set_title(f'Weighted geomedian (NDVI, rho={rho})')
ax[2].set_title(f'Weighted geomedian (NDVI, rho={-rho})')
for a in ax.ravel():
    a.set(xticks=[], yticks=[])

### Plot NDVI

In [ ]:
def ndvi(nir, red):
    return (nir - red) / (nir+red)

In [ ]:
fig,ax=plt.subplots(1,3, figsize=(15,5), sharey=True, layout='constrained')
ndvi(gm.nbart_nir_1, gm.nbart_red).plot(ax=ax[0], vmin=0, vmax=0.9, cmap='RdYlGn', add_labels=False);
ndvi(wgm_green.nbart_nir_1, wgm_green.nbart_red).plot(ax=ax[1], vmin=0, vmax=0.9, cmap='RdYlGn', add_labels=False)
ndvi(wgm_bare.nbart_nir_1, wgm_bare.nbart_red).plot(ax=ax[2], vmin=0, vmax=0.9, cmap='RdYlGn', add_labels=False)
ax[0].set_title(f'Standard geomedian')
ax[1].set_title(f'Weighted geomedian NDVI, rho={rho})')
ax[2].set_title(f'Weighted geomedian NDVI, rho={-rho})')
for a in ax.ravel():
    a.set(xticks=[], yticks=[])

***

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Australia data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Discord chat](https://discord.com/invite/4hhBQVas5U) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [GitHub](https://github.com/GeoscienceAustralia/dea-notebooks).

**Last modified:** July 2026

**Compatible datacube version:** 

In [ ]:
print(datacube.__version__)